# Part 3: Data structures

**Goal:** Store messages and turn messy JSON replies into useful dictionaries.

**Time:** 30 minutes. **Run cells top to bottom.**

First, make sure the two small libraries used in the examples are installed. Your local environment usually already has them; Colab may need them.

In [ ]:
import importlib.util
import subprocess
import sys
missing = [pkg for module, pkg in [("requests", "requests"),
           ("dotenv", "python-dotenv")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

Find the workshop folder. In Colab, download a fresh copy if needed. This also makes `starter/` imports and file paths work in every notebook.

In [ ]:
import os
from pathlib import Path
import tempfile
root = Path.cwd()
if not (root / "starter").exists() and (root.parent / "starter").exists():
    root = root.parent
if not (root / "starter").exists():
    root = Path(tempfile.mkdtemp()) / "workshop"
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/HruthikKommuru/cwru-fall26-session7.git", str(root)])
os.chdir(root)
sys.path.insert(0, str(root))

A list holds messages in order. Each message is a dictionary with named fields. Index `1` means the second item because Python counts from zero.

In [ ]:
messages = [{"role": "system", "content": "Be concise."},
            {"role": "user", "content": "Check this clause."}]
print(messages[1]["content"])
messages.append({"role": "assistant", "content": "I will check it."})
print(messages)

JSON is text used to exchange data. `dumps` turns a Python dictionary into text; `loads` turns valid JSON text back into Python data.

In [ ]:
import json
original = {"title": "Refund policy", "score": 8}
print(type(original))
reply = json.dumps(original)
print(type(reply))
restored = json.loads(reply)
print(type(restored))

Catch a specific decoding error when text is incomplete. `.get()` reads a field with a fallback. This small example assumes any parsed value is a dictionary.

In [ ]:
reply = '{"title": "Refund policy"'
try:
    data = json.loads(reply)
except json.JSONDecodeError:
    data = None
title = (data or {}).get("title", "not stated")
print(title)

## Five messy replies

| Problem | A practical fix for these samples |
| --- | --- |
| Markdown JSON fences | Start decoding at the first opening brace. |
| Chat before and after JSON | Decode one object and ignore surrounding chat. |
| Missing score | Use the fallback `"not stated"`. |
| Score is `"8"` | Convert the numeric text with `int()`. |
| Cut-off JSON | Catch the decoding error and return `None`. |

## Your turn — Lab 3

Write `parse_reply(text)` for all five supplied samples. Samples 1, 2, and 4
should return `{"title": "Refund policy", "score": 8}`. Sample 3 should return
`{"title": "Refund policy", "score": "not stated"}`. Sample 5 should return `None`.

Hint: find `text.find("{")`, then use
`json.JSONDecoder().raw_decode(text[start:])` to read one JSON value. It returns
the parsed object and the position where parsing stopped. Check `isinstance(data, dict)`.
This is a teaching parser for these five formats, not a guarantee for arbitrary model text.

Load the sample list and create your parser. Replace `parse_reply = None` with a function definition; use specific error handling for invalid JSON.

In [ ]:
from starter.replies import REPLIES
parse_reply = None
# TODO: def parse_reply(text): ...
# TODO: find the object, decode it, fill missing fields, convert score.
# TODO: return None when the object cannot be parsed.

**Check yourself:** compare all five outputs to the expected dictionaries. The loop reports which numbered sample needs another look.

In [ ]:
expected = [{"title": "Refund policy", "score": 8}] * 2
expected += [{"title": "Refund policy", "score": "not stated"}]
expected += [{"title": "Refund policy", "score": 8}, None]
if parse_reply is None:
    print("Write parse_reply, then run this check again.")
else:
    for number, (text, wanted) in enumerate(zip(REPLIES, expected), 1):
        assert parse_reply(text) == wanted, f"Revisit sample {number}"
    print("Nice work! All five replies passed.")

**Break it:** invent another reply that defeats your parser. Look for braces inside chat text or an unexpected score type. Explain the result in your own words.

In [ ]:
my_reply = 'Note {draft}: {"title": "Refund policy", "score": 8}'
# TODO: replace this with your own tricky reply.
if parse_reply is not None:
    print(parse_reply(my_reply))

## What you learned

- Lists preserve order and dictionaries name fields.
- JSON text and Python dictionaries are different types.
- Parsing needs checks, fallback values, and a clear failure result.